In [ ]:
import requests
import csv
from datetime import datetime, timedelta
import time
import json
from dotenv import load_dotenv
import os

load_dotenv()

# --- Configuration ---
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")
user_agent = os.getenv("USER_AGENT")
username = os.getenv("USERNAME")
password = os.getenv("PASSWORD")
subreddit = 'politics'
query = 'russian AND invasion' #putin #zelensky #trump #trump and zelensky #war AND ukraine
comment_score_min = 0
output_json = 'russian_invasion_rel_p.json'

# --- SETTINGS ---
max_pages = 50
request_delay = 1.5
limit_per_page = 1000

# --- Step 1: OAuth2 Token ---
print("🔐 Obtaining access token...")
auth = requests.auth.HTTPBasicAuth(client_id, client_secret)
data = {'grant_type': 'password', 'username': username, 'password': password}
headers = {'User-Agent': user_agent}

try:
    res = requests.post('https://www.reddit.com/api/v1/access_token', auth=auth, data=data, headers=headers)
    token = res.json()['access_token']
    headers['Authorization'] = f'bearer {token}'
    print("✅ Token obteined!")
except Exception as e:
    print(f"❌ Error while obtaing token: {e}")
    exit(1)

# --- Helper: Recursive Comment Extractor ---
def extract_comments(children, threshold):
    results = []
    for child in children:
        kind = child.get('kind')
        data = child.get('data', {})
        if kind != 't1':
            continue
        score = data.get('score', 0)
        body = data.get('body', '')
        if score >= threshold and not body.lower().startswith('[deleted') and body.strip():
            results.append({
                'author': data.get('author'),
                'score': score,
                'body': body,
                'created_utc': datetime.utcfromtimestamp(data['created_utc']).isoformat()
            })
        replies = data.get('replies')
        if replies and isinstance(replies, dict):
            results.extend(extract_comments(replies['data']['children'], threshold))
    return results

# --- Main Loop with "after" ---
all_data = []
total_posts = 0
total_comments = 0
current_page = 1
after_token = None

print("\n📄 Starting data collection...")
print(f"🔍 Query: '{query}' in r/{subreddit}")

while current_page <= max_pages:
    print(f"\n📃 Page {current_page}/{max_pages}...")
    
    # Parameters
    search_url = f'https://oauth.reddit.com/r/{subreddit}/search'
    params = {
        'q': query,
        'limit': limit_per_page,
        'sort': 'relevance',
        'restrict_sr': True
    }
    
    # Add after token (if available)
    if after_token:
        params['after'] = after_token
    
    try:
        resp = requests.get(search_url, headers=headers, params=params)
        
        if resp.status_code != 200:
            print(f"⚠️ Error API: {resp.status_code}")
            print(f"Response: {resp.text}")
            break
            
        data = resp.json()
        posts = data['data']['children']
        after_token = data['data'].get('after')
        
        print(f"   📝 Found {len(posts)} posts in this page")
    
        if not posts:
            print("   ℹ️ No post found, finish")
            break
            
        if not after_token:
            print("   ℹ️ Last page reached (no 'after' token)")
        
        page_posts = 0
        page_comments = 0
        
        # Process each post of page
        for post in posts:
            post_data = post['data']
            post_id = post_data['id']
            title = post_data['title']
            selftext = post_data.get('selftext', '')
            score = post_data['score']
            author = post_data.get('author', '[deleted]')
            created_utc = datetime.utcfromtimestamp(post_data['created_utc']).isoformat()
            num_comments = post_data['num_comments']
            
            # Fetch comments for this post
            comment_url = f'https://oauth.reddit.com/comments/{post_id}.json'
            try:
                response = requests.get(comment_url, headers=headers, params={'depth': 10, 'limit': 500})
                if response.status_code == 200:
                    comment_blob = response.json()[1]['data']['children']
                    high_comments = extract_comments(comment_blob, comment_score_min)
                else:
                    print(f"      ⚠️ Error {response.status_code} for comments post {post_id}")
                    high_comments = []
            except Exception as e:
                print(f"      ⚠️ Errore comments for post {post_id}: {e}")
                high_comments = []
            
            # Build record
            post_record = {
                'page_number': current_page,
                'post_id': post_id,
                'title': title,
                'author': author,
                'score': score,
                'created_utc': created_utc,
                'selftext': selftext,
                'num_comments': num_comments,
                'high_score_comments': high_comments
            }
            
            all_data.append(post_record)
            page_posts += 1
            page_comments += len(high_comments)

            time.sleep(0.1)
        
        total_posts += page_posts
        total_comments += page_comments
        
        print(f"   ✅ {page_posts} posts, {page_comments} comments collected from page {current_page}")
        print(f"   📊 Total until now: {total_posts} posts, {total_comments} comments")

        if not after_token:
            break
            
    except Exception as e:
        print(f"❌ Error page {current_page}: {e}")
        break
    
    current_page += 1

    time.sleep(request_delay)

    if current_page % 10 == 0:
        print(f"💾 Save progress... ({total_posts} posts until now)")
        with open(f"temp_{output_json}", 'w', encoding='utf-8') as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)

# --- Final save ---
print(f"\n💾 Final save...")
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

# Final Statistics
print(f"\n🎉 COMPLETED!")
print(f"📊 FINAL STATISTICS:")
print(f"   📄 Processed pages: {current_page - 1}")
print(f"   📝 Total posts: {total_posts}")
print(f"   💬 Total comments: {total_comments}")
print(f"   📄 File saved: {output_json}")
if current_page > 1:
    print(f"   ⏱️ Average posts/page: {total_posts/(current_page-1):.1f}")

if os.path.exists(f"temp_{output_json}"):
    os.remove(f"temp_{output_json}")
    print(f"🗑️ Temporary file removed")

print(f"\n✨ Collection completed! Data found from {len(set(record['post_id'] for record in all_data))} unique posts.")

🔐 Obtaining access token...
❌ Error while obtaing token: HTTPSConnectionPool(host='www.reddit.com', port=443): Max retries exceeded with url: /api/v1/access_token (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000262E0FE6590>: Failed to resolve 'www.reddit.com' ([Errno 11001] getaddrinfo failed)"))

📄 Starting data collection...
🔍 Query: 'russian AND invasion' in r/politics

📃 Page 1/50...
❌ Error page 1: HTTPSConnectionPool(host='oauth.reddit.com', port=443): Max retries exceeded with url: /r/politics/search?q=russian+AND+invasion&limit=1000&sort=relevance&restrict_sr=True (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000262E1106350>: Failed to resolve 'oauth.reddit.com' ([Errno 11001] getaddrinfo failed)"))

💾 Final save...

🎉 COMPLETED!
📊 FINAL STATISTICS:
   📄 Processed pages: 0
   📝 Total posts: 0
   💬 Total comments: 0
   📄 File saved: russian_invasion_rel_p.json

✨ Collection completed! Data found from 0 uniq

In [ ]:
import json
import pandas as pd

FILENAME = "russian_invasion_rel_p"

# --- Load JSON file ---
with open(f'{FILENAME}.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# --- Statistics ---
num_posts = len(data)
num_comments = sum(len(post['high_score_comments']) for post in data)

print(f"📌 Total number of posts: {num_posts}")
print(f"💬 Total number of high-score comments: {num_comments}")

# --- Create DataFrame ---
rows = []
for post in data:
    post_id = post['post_id']
    post_title = post['title']
    post_author = post['author']
    post_score = post['score']
    post_created_utc = post['created_utc']
    
    for comment in post['high_score_comments']:
        rows.append({
            'post_id': post_id,
            'post_title': post_title,
            'post_author': post_author,
            'post_score': post_score,
            'post_created_utc': post_created_utc,
            'comment_author': comment['author'],
            'comment_score': comment['score'],
            'comment_body': comment['body'],
            'comment_created_utc': comment['created_utc']
        })

df = pd.DataFrame(rows)

# --- Preview DataFrame ---
print("\n🧾 DataFrame preview (last rows):")
df.tail()

# Save as CSV if needed
df.to_csv(f'{FILENAME}.csv', index=False)